# 13｜Valuation P0＋P1 實際盤點與財務科目名稱 Mapping

**Research only** / `publication_allowed=false` / 不修改 `canonical/CURRENT.json`、不抓新資料、不訓練、不部署。

固定 GitHub `tommylin15/janus-omniforge` `main` implementation commit：`dca20eb573f091b4925f659045809ff6f1072bbb`。

本 Notebook **不內嵌 Python 模組**；僅載入 `janusChatGPT/CoLab程式` 兩個 Python 同版檔並比對 SHA256。沒有檔案或雜湊不一致即停止，避免默默執行舊版本。只讀 CURRENT 指向的財報 Parquet，輸出名稱映射與 P0/P1 研究收據。`RUN_FETCH=false`。


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q','pandas==2.2.3','pyarrow==20.0.0'])


In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os, json, hashlib, importlib

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/給ai專用/janusChatGPT')
assert ROOT.is_dir(), '原授權 janusChatGPT 根目錄不存在；禁止改用「個人用」或建立同名資料夾'
CODE_DIR = ROOT / 'CoLab程式'
GIT_IMPLEMENTATION_SHA = 'dca20eb573f091b4925f659045809ff6f1072bbb'
PINNED_SHA256 = {
    'valuation_backfill.py': 'db474a1cd8c17b9dd28ff9e3c17485d104f47311b927eae73e1c2cec08cc42c5',
    'valuation_mapping_p0p1.py': '3d28a56f9c279babc7732c73a1b619033726b6831157db84c8e2faea0fdb46f1',
}
for name, expected_sha in PINNED_SHA256.items():
    f = CODE_DIR / name
    if not f.is_file(): raise FileNotFoundError('請先從 GitHub 對應 commit 同步檔案到 CoLab程式: '+name)
    actual_sha = hashlib.sha256(f.read_bytes()).hexdigest()
    if actual_sha != expected_sha: raise ValueError('MODULE_SHA256_MISMATCH: '+name)
if str(CODE_DIR) not in sys.path: sys.path.insert(0,str(CODE_DIR))
for mod in ('valuation_backfill','valuation_mapping_p0p1'): sys.modules.pop(mod,None)
importlib.invalidate_caches()
from valuation_backfill import checked_root, checked_current
from valuation_mapping_p0p1 import live_canonical_audit, MAPPING_VERSION
checked_root(ROOT)
print('Verified module SHA256, pinned Git SHA:', GIT_IMPLEMENTATION_SHA)


In [ ]:
# 本 Notebook 只盤點與對應，不設任何 HTTP 補收呼叫
DRY_RUN = True
RUN_FETCH = False
assert DRY_RUN and not RUN_FETCH
RUN_ID = 'P0P1_20261024'     # 如需獨立版本可變更此 ID
OUTPUT = ROOT/'模型研究'/'valuation'/'backfill_candidates'/RUN_ID/'p0p1'
print('Output:', OUTPUT)


## P0＋P1：完整讀回與報表
執行此格會驗證現有 canonical manifest 指向的檔案；不修改來源，不發任何 HTTP API 補收。


In [ ]:
# 逐一讀回 CURRENT 的全部 references SHA256，再讀 15 季財報 Parquet
# 不需 API token。成功產生報告不能等同 PIT/單位認證。
summary = live_canonical_audit(ROOT, OUTPUT, verify_manifest=checked_current)
assert summary['publication_allowed'] is False and summary['HTTP_requests_executed']==0
print(json.dumps(summary,ensure_ascii=False,indent=2))


In [ ]:
from IPython.display import display
import pandas as pd
for name in ['financial_mapping_audit.csv','field_coverage.csv','factor_readiness.csv','coverage_by_dataset_status.csv']:
    file=OUTPUT/name
    print('\n',name,'found=',file.is_file())
    if file.is_file():display(pd.read_csv(file).head(12))
print('inventory.gz:', OUTPUT/'inventory.csv.gz')
print('gap_plan.csv:', OUTPUT/'gap_plan.csv')
print('Notes: 0 HTTP requests; mapping labels only, no safe financial factors until scope/unit/PIT verified.')


## 停止／重試
若 Drive 暫時中斷，重新掛載後從第 3 格執行；P0 盤點會重新讀既有 Parquet，**不會發生重複 HTTP 補收**。未成功讀完不得稱完成。大報表保留於既有 `模型研究/valuation/backfill_candidates/<RUN_ID>/p0p1`。
